# densepoly: alur penuh, dari data sampai tabel, termasuk training dari awal

Notebook ini menjalankan **seluruh alur** proyek untuk eksperimen yang dipilih di sel pertama: pemeriksaan data, ground truth
evaluasi, training dari bobot ImageNet, tuning post-processing di `val_tune_sub`, prediksi dan evaluasi di `test`, lalu tabel dan
contoh gambar.

Notebook ini tipis: setiap langkah memanggil skrip di `scripts/`, dan semua logika ada di `src/densepoly/`. Angka yang tampil dibaca
dari file keluaran skrip, bukan diketik.

**Keluaran ditulis ke folder tersendiri** (`OUT`), supaya run dan checkpoint yang sudah dibekukan tidak tertimpa. File tuning yang
dibuat di sini juga disimpan di salinan `configs/` di dalam `OUT`, bukan di repo.

**Biaya.** Satu run FULL-P0 (E = 60) butuh ±2 jam di RTX 5090. Semua run proyek (±18 run) butuh ±33 jam GPU.

In [ ]:
# Settings of this notebook
DATA = "crowdai"
RUNS = [("full_p0", 0)]   # (experiment, seed) pairs to train from scratch, e.g. [("b4", 0), ("full_p0", 0)]
EPOCHS = 60               # E, the same for every CrowdAI run (docs/EXPERIMENTS.md §3)
TRAIN_WORKERS = 4         # loader processes of the training
CPU_WORKERS = 8           # processes of the tuning and the evaluation
RUN_DATA_PREP = False     # True also rebuilds the instance cache, the splits and the density strata from the raw download
OUT = ""                  # output folder; empty: $DENSEPOLY_OUT_FULL, else <repo>/outputs_full_pipeline
REFERENCE_OUT = ""        # outputs of the frozen runs to compare with; empty: $DENSEPOLY_REFERENCE_OUT, else no comparison

## 1. Lingkungan dan folder keluaran

Sel ini menetapkan folder keluaran, menyalin `configs/` ke sana (tanpa file tuning eksperimen yang akan dilatih), dan menyiapkan
fungsi `run` yang menjalankan satu skrip, menyimpan log lengkapnya, dan berhenti bila skrip gagal.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)

out = Path(OUT or os.environ.get("DENSEPOLY_OUT_FULL") or REPO / "outputs_full_pipeline").resolve()
reference = REFERENCE_OUT or os.environ.get("DENSEPOLY_REFERENCE_OUT") or ""
frozen = os.environ.get("DENSEPOLY_OUT")
assert frozen is None or Path(frozen).resolve() != out, "OUT must not be the folder of the frozen runs"
os.environ["DENSEPOLY_OUT"] = str(out)

configs = out / "configs"  # the tunings of this notebook go here, never into the repo
if not configs.exists():
    shutil.copytree(REPO / "configs", configs)
    for exp, _ in RUNS:
        (configs / "postproc" / DATA / f"{exp}.yaml").unlink(missing_ok=True)
logs = out / "logs"
logs.mkdir(parents=True, exist_ok=True)


def run(name, *args, tail=6):
    """Run a script of the repo with this Python, keep its whole output in logs/<name>.log, show the last lines."""
    command = [sys.executable, *map(str, args)]
    print("$ python", " ".join(map(str, args)), flush=True)
    tic = time.perf_counter()
    with open(logs / f"{name}.log", "w", encoding="utf-8") as log:
        status = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT).returncode
    lines = (logs / f"{name}.log").read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(line[:300] for line in lines[-tail:]))
    print(f"[{name}: {time.perf_counter() - tic:.0f} s, exit {status}]", flush=True)
    assert status == 0, f"{name} failed: see {logs / (name + '.log')}"


print("repo     ", REPO)
print("out      ", out)
print("data     ", os.environ.get("DENSEPOLY_DATA", REPO / "data"))
print("reference", reference or "(none)")

In [ ]:
import torch

from densepoly import paths
from densepoly.utils.provenance import git_revision

print("python", sys.version.split()[0], "| torch", torch.__version__, "| git", git_revision()[:12])
assert torch.cuda.is_available(), "the training needs a GPU"
print("GPU", torch.cuda.get_device_name(0))
assert paths.dataset_dir(DATA).exists(), f"{paths.dataset_dir(DATA)} does not exist: download the dataset first (docs/DATA.md §1)"

## 2. Data

Split, strata kepadatan, dan cache peta instance sudah dibekukan sejak Fase 1 (`splits/`, `docs/DATA.md`). Membuatnya ulang dari
unduhan mentah memakan berjam-jam CPU, jadi langkah itu hanya berjalan bila `RUN_DATA_PREP = True`. Sel berikutnya selalu
memeriksa bahwa semua tile dan peta instance yang dipakai memang ada.

In [ ]:
if RUN_DATA_PREP:
    run("prepare_stats", "scripts/prepare_crowdai.py", "--stats")
    run("dedup", "scripts/dedup_crowdai.py", "--data", DATA, "--workers", CPU_WORKERS)
    run("prepare_cache", "scripts/prepare_crowdai.py", "--cache", "train_dedup", "val_tune", "test")
    run("density", "scripts/make_density.py", "--data", DATA, "--split", "test")
else:
    print("data preparation skipped: the frozen splits and caches are used")

In [ ]:
from densepoly.data.dataset import tile_refs

for split in ("train_dedup", "val_select", "val_tune_sub", "test"):
    refs = tile_refs(DATA, split)
    missing_images = sum(not ref.image.exists() for ref in refs)
    missing_maps = sum(not paths.instance_cache(DATA, split, ref.key).exists() for ref in refs)
    print(f"{split:13s} {len(refs):6d} tiles, missing images {missing_images}, missing instance maps {missing_maps}")
    assert missing_images == 0 and missing_maps == 0

## 3. Ground truth evaluasi

Ground truth COCO per split ditulis ke `OUT/gt/`. Evaluasi hanya pernah membaca file ini.

In [ ]:
run("prepare_gt", "scripts/prepare_gt.py", "--data", DATA, "--split", "val_tune_sub", "val_select", "test")

## 4. Training dari awal

Sebelum training, kecepatan diukur 200 iterasi. Lalu setiap run dilatih `EPOCHS` epoch dari bobot ImageNet. Checkpoint terbaik
dipilih dengan IoU foreground di `val_select`. `--resume` membuat sel ini aman dijalankan ulang: run yang sudah selesai dilewati.

In [ ]:
for exp, seed in RUNS:
    run(f"benchmark_{exp}", "scripts/train.py", "--data", DATA, "--exp", exp, "--benchmark", 200,
        "--configs-dir", configs, f"train.num_workers={TRAIN_WORKERS}", tail=3)

In [ ]:
for exp, seed in RUNS:
    run(f"train_{exp}_seed{seed}", "scripts/train.py", "--data", DATA, "--exp", exp, "--seed", seed, "--resume",
        "--configs-dir", configs, f"train.epochs={EPOCHS}", f"train.num_workers={TRAIN_WORKERS}", tail=3)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

for exp, seed in RUNS:
    run_dir = paths.run_dir(DATA, exp, paths.run_name(seed))
    train_log, val_log = pd.read_csv(run_dir / "train_log.csv"), pd.read_csv(run_dir / "val_log.csv")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    smooth = train_log.set_index("step")[["loss", "ce", "dice", "theta"]].rolling(40, min_periods=1).mean()
    smooth.plot(ax=axes[0], linewidth=1.2)
    axes[0].set(title=f"{exp} seed {seed}: training loss (rolling mean)", xlabel="step")
    val_log.plot(x="epoch", y="fg_iou", ax=axes[1], marker="o", markersize=3, legend=False)
    axes[1].set(title="foreground IoU on val_select", xlabel="epoch")
    plt.tight_layout()
    plt.show()
    best = val_log.loc[val_log["fg_iou"].idxmax()]
    print(f"best epoch {int(best['epoch'])}: fg IoU {best['fg_iou']:.4f}; training time {val_log['seconds'].iloc[-1] / 3600:.2f} h")

## 5. Tuning post-processing di `val_tune_sub`

Keluaran jaringan di `val_tune_sub` disimpan sekali, lalu grid yang sama untuk semua model (`configs/postproc/grid.yaml`) dicoba di
atasnya. Nilai terpilih dibekukan sebelum `test` disentuh.

In [ ]:
from omegaconf import OmegaConf

for exp, seed in RUNS:
    run(f"predict_{exp}_val_tune_sub", "scripts/predict.py", "--data", DATA, "--exp", exp, "--seed", seed,
        "--split", "val_tune_sub", "--save-channels", "--configs-dir", configs, tail=1)
    tuned = configs / "postproc" / DATA / f"{exp}.yaml"
    if not tuned.exists():
        run(f"tune_{exp}", "scripts/tune_postproc.py", "--data", DATA, "--exp", exp, "--workers", CPU_WORKERS,
            "--configs-dir", configs, tail=1)
    frozen_tuning = OmegaConf.to_container(OmegaConf.load(tuned))
    print(exp, "frozen:", frozen_tuning["postproc"], "| val_tune_sub:", frozen_tuning["tuning"]["score"])

## 6. Prediksi dan evaluasi di `test`

`test` adalah 10% klaster train yang disisihkan sejak awal. Tidak ada parameter yang disetel di sini.

In [ ]:
for exp, seed in RUNS:
    run(f"predict_{exp}_test", "scripts/predict.py", "--data", DATA, "--exp", exp, "--seed", seed, "--split", "test",
        "--configs-dir", configs, tail=1)
    run(f"evaluate_{exp}_test", "scripts/evaluate.py", "--data", DATA, "--exp", exp, "--seed", seed, "--split", "test",
        "--workers", CPU_WORKERS, tail=2)

## 7. Tabel hasil

Tabel dibuat `scripts/tables.py` dari file metrik di `OUT/metrics/`. Tabel yang butuh tiga seed (N1, N3) hanya muncul bila ketiga
seed-nya dilatih di notebook ini.

In [ ]:
from IPython.display import Markdown, display

run("tables", "scripts/tables.py", "--data", DATA, "--table", "ablation", "strata", tail=2)
display(Markdown((out / "tables" / DATA / "ablation.md").read_text(encoding="utf-8")))

In [ ]:
SHOWN = ("AP", "AP50", "AP_boundary", "IoU", "CIoU", "PoLiS", "MTA", "PoLiS_paired", "MTA_inst", "merge_rate", "split_rate")
rows = {}
for exp, seed in RUNS:
    name = f"{exp} seed {seed}"
    rows[f"{name} (this notebook)"] = json.loads(paths.metrics_file(DATA, exp, paths.run_name(seed), "test").read_text())["overall"]
    if reference:
        frozen_file = Path(reference) / "metrics" / DATA / exp / paths.run_name(seed) / "test.json"
        if frozen_file.exists():
            rows[f"{name} (frozen run)"] = json.loads(frozen_file.read_text())["overall"]
comparison = pd.DataFrame({name: {metric: row[metric] for metric in SHOWN} for name, row in rows.items()})
display(comparison.style.format("{:.4f}"))

Dua run dengan seed yang sama tidak identik per bit (operasi GPU tidak deterministik), jadi selisih kecil terhadap run beku itu
wajar. Yang perlu dicek: selisihnya sebesar variasi antar-seed, bukan lebih.

## 8. Contoh hasil di tile padat

Empat tile `test` dari strata padat: batas ground truth (hijau) dan poligon prediksi (kuning).

In [ ]:
import numpy as np

from densepoly.priors.cache import read_rgb
from densepoly.utils.io import read_json

exp, seed = RUNS[0]
dense = read_json(paths.density_file(DATA, "test"))["main"]["strata"]["padat"]
picked = set(np.random.default_rng(0).choice(dense, 4, replace=False).tolist())
gt = read_json(paths.gt_file(DATA, "test"))
predictions = read_json(paths.pred_file(DATA, exp, paths.run_name(seed), "test"))
files = {image["id"]: image["file_name"] for image in gt["images"] if image["id"] in picked}

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, image_id in zip(axes, sorted(picked)):
    ax.imshow(read_rgb(paths.image_dir(DATA, "test") / files[image_id]))
    for source, colour, width in ((gt["annotations"], "#39d353", 1.2), (predictions, "#ffd60a", 1.6)):
        for item in source:
            if item["image_id"] == image_id:
                ring = np.reshape(item["segmentation"][0], (-1, 2))
                ax.plot(*np.vstack([ring, ring[:1]]).T, color=colour, linewidth=width)
    ax.set_title(f"tile {image_id}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Ringkasan

- Semua keluaran notebook ini ada di `OUT`: `runs/` (checkpoint dan log training), `configs/postproc/` (tuning beku), `preds/`,
  `metrics/`, `tables/`, dan `logs/` (log lengkap tiap skrip).
- Untuk melatih eksperimen lain atau seed lain, ubah `RUNS` di sel pertama lalu jalankan ulang. Langkah yang sudah selesai dilewati
  oleh `--resume` dan oleh pemeriksaan file tuning.
- Versi berisi keluaran diekspor dengan
  `jupyter nbconvert --to html --execute notebooks/full_pipeline.ipynb --output-dir <OUT>/report`.